In [2]:
import pandas as pd
import numpy as np
import pandas as pd
import numpy as np
import os
import warnings
from datetime import datetime


def load_spec(specfile):
    """
    Load model specification for a dynamic factor model (DFM).

    Parameters:
    - specfile: str, path to the Excel file containing the model specification.

    Returns:
    - spec: dict, containing the model specification.
    """

    # Read the Excel file
    raw_data = pd.read_excel(specfile, sheet_name=None, header=None)
    raw_data = list(raw_data.values())[0]  # Get the first sheet data if multiple sheets exist
    raw_data.columns = raw_data.iloc[0].str.replace(' ', '')
    raw_data = raw_data.drop(0)

    # Convert all headers to lowercase for consistency
    raw_data.columns = raw_data.columns.str.lower()

    # Find and drop series from Spec that are not in Model
    model_idx = raw_data['model'].astype(int) != 0
    raw_data = raw_data[model_idx]

    # Initialize spec dictionary
    spec = {}

    # Fields to extract from the Excel file
    field_names = ['seriesid', 'seriesname', 'frequency', 'units', 'transformation', 'category']
    for field in field_names:
        if field in raw_data.columns:
            spec[field] = raw_data[field].tolist()
        else:
            raise ValueError(f"{field} column missing from model specification.")

    # Parse blocks
    block_cols = [col for col in raw_data.columns if col.startswith('block')]
    blocks = raw_data[block_cols].fillna(0).astype(int).values
    if not np.all(blocks[:, 0] == 1):
        raise ValueError('All variables must load on global block.')
    spec['blocks'] = blocks

    # Sort all fields of 'Spec' in order of decreasing frequency
    frequency_order = ['d', 'w', 'm', 'q', 'sa', 'a']
    permutation = []
    for freq in frequency_order:
        permutation.extend(np.where(np.array(spec['frequency']) == freq)[0])
    for field in spec.keys():
        spec[field] = [spec[field][i] for i in permutation]

    # Extract block names from header
    spec['blocknames'] = [col.replace('block', '').replace('-', '') for col in block_cols]

    # Transformations
    transformation_map = {
        'lin': 'Levels (No Transformation)',
        'chg': 'Change (Difference)',
        'ch1': 'Year over Year Change (Difference)',
        'pch': 'Percent Change',
        'pc1': 'Year over Year Percent Change',
        'pca': 'Percent Change (Annual Rate)',
        'cch': 'Continuously Compounded Rate of Change',
        'cca': 'Continuously Compounded Annual Rate of Change',
        'log': 'Natural Log'
    }
    spec['unitstransformed'] = [
        transformation_map.get(trans, trans) for trans in spec['transformation']
    ]
    # Summarize model specification
    print('Table 1: Model specification')
    try:
        tabular = pd.DataFrame({
            'SeriesID': spec['seriesid'],
            'SeriesName': spec['seriesname'],
            'Units': spec['units'],
            'Transformation': spec['unitstransformed']
        })
        print(tabular)
    except Exception as e:
        print(f"Failed to display table: {e}")

    return spec


In [3]:
def load_data(ds, Spec, sample=None, load_excel=False):
    """
    Load vintage of data from file and format as structure

    Parameters:
        datafile (str): Filename of Microsoft Excel workbook file
        Spec (dict): Model specification containing SeriesID and other info
        sample (float, optional): Sample period start date in numeric form
        load_excel (bool, optional): Flag to force loading from Excel

    Returns:
        X (np.ndarray): T x N numeric array, transformed dataset
        Time (np.ndarray): T x 1 numeric array, date number with observation dates
        Z (np.ndarray): T x N numeric array, raw (untransformed) dataset
    """
    print('Loading data...')

    Z, Time, Mnem = read_data(ds)

    # Sort data based on model specification
    Z = sort_data(Z, Mnem, Spec)
    
    # Transform data based on model specification
    X, Time, Z, header = transform_data(Z, Time, Spec)

    # Drop data not in estimation sample
    if sample is not None:
        X, Time, Z = drop_data(X, Time, Z, sample)

    # Z = np.vstack([header, Z])
    # X = np.vstack([header, X])

    return X, Time, Z, header


def read_data(ds):
    """
    Read data from Microsoft Excel workbook file

    Parameters:
        datafile (str): Filename of the Excel file

    Returns:
        Z (np.ndarray): Raw (untransformed) observed data
        Time (np.ndarray): Observation periods for the time series data
        Mnem (list): Series ID for each variable
    """
    # df = pd.read_excel(datafile, sheet_name='data', header=None, engine="openpyxl")
    Mnem = ds.iloc[0, 1:].tolist()

    # if os.name == 'nt':  # Check if the operating system is Windows
    #     Time = pd.to_datetime(df.iloc[1:, 0], format='%m/%d/%Y').astype(np.int64) // 10**9
    #     Z = df.iloc[1:, 1:].to_numpy()
    # else:
    #     Time = (df.iloc[1:, 0] + pd.Timestamp('1899-12-31').to_julian_date()).to_numpy()
    #     Z = df.iloc[:, 1:].to_numpy()
    Time = ds.iloc[1:, 0].to_numpy()
    Z = ds.iloc[:, 1:].to_numpy()
    return Z, Time, Mnem


def sort_data(Z, Mnem, Spec):
    """
    Sort series by order of model specification

    Parameters:
        Z (np.ndarray): Raw data
        Mnem (list): Series ID for each variable
        Spec (dict): Model specification

    Returns:
        Z (np.ndarray): Sorted data according to Spec.SeriesID
    """
    in_spec = np.isin(Mnem, Spec['seriesid'])
    Mnem = [mnem for mnem, keep in zip(Mnem, in_spec) if keep]
    Z = Z[:, in_spec]

    # Sort series by ordering of Spec
    N = len(Spec['seriesid'])
    permutation = [Mnem.index(spec_id) for spec_id in Spec['seriesid']]

    Mnem = [Mnem[i] for i in permutation]
    Z = Z[:, permutation]

    return Z


def transform_data(Z, Time, Spec):
    """
    Transforms each data series based on Spec.Transformation

    Parameters:
        Z (np.ndarray): Raw (untransformed) observed data
        Time (np.ndarray): Observation periods for the time series data
        Spec (dict): Model specification

    Returns:
        X (np.ndarray): Transformed data (stationary to enter DFM)
        Time (np.ndarray): Adjusted time data
        Z (np.ndarray): Adjusted raw data
    """
    header = Z[0, :]
    Z = np.float64(Z[1:, :])

    T, N = Z.shape

    X = np.full((T, N), np.nan)

    for i in range(N):
        formula = Spec["transformation"][i]
        freq = Spec["frequency"][i]
        step = 1 if freq == "m" else 3
        t1 = step
        n = step / 12

        assert header[i]== Spec["seriesid"][i]
        series = Spec["seriesname"][i]
        
         # Apply transformations based on formula
        if formula == 'lin':  # Levels (No Transformation)
            X[:, i] = Z[:, i]
        elif formula == 'chg':  # Change (Difference)
            X[t1:T:step, i] = np.concatenate(([np.nan], Z[(t1+step):T:step, i] - Z[t1:(T-t1):step, i]))
        elif formula == 'ch1':  # Year over Year Change (Difference)
            if T > 12:
                X[(12+t1):T:step, i] = Z[(12+t1):T:step, i] - Z[t1:(T - 12):step, i]
        elif formula == 'pch':  # Percent Change
            X[t1:T:step, i] = 100 * np.concatenate(
                ([np.nan], Z[(t1+step):T:step, i] / Z[t1:(T-t1):step, i] - 1)
            )
        elif formula == 'pc1':  # Year over Year Percent Change
            if T > 12:
                # Year over Year Percent Change, handle division by zero
                X[(12+t1):T:step, i] = 100 * (Z[(12+t1):T:step, i] / Z[t1:(T-12):step, i] - 1)
        elif formula == 'pca':  # Percent Change (Annual Rate)
            X[t1:T:step, i] = 100 * np.concatenate(
                ([np.nan], (Z[(t1+step):T:step, i] / Z[t1:(T-step):step, i]) ** (1 / n) - 1)
            )
        elif formula == 'log':  # Natural Log
            X[:, i] = np.log(Z[:, i])
        else:
            warnings.warn(f"Transformation '{formula}' not found for {series}. Using untransformed data.")
            X[:, i] = Z[:, i]

    # Drop first quarter of observations since transformations cause missing values
    Time = Time[3:]
    Z = Z[3:, :]
    X = X[3:, :]

    return X, Time, Z, header


def drop_data(X, Time, Z, sample):
    """
    Remove data not in estimation sample

    Parameters:
        X (np.ndarray): Transformed data
        Time (np.ndarray): Time data
        Z (np.ndarray): Raw data
        sample (float): Sample period start date in numeric form

    Returns:
        X (np.ndarray): Filtered transformed data
        Time (np.ndarray): Filtered time data
        Z (np.ndarray): Filtered raw data
    """
    idx_drop = Time < sample

    Time = Time[~idx_drop]
    X = X[~idx_drop, :]
    Z = Z[~idx_drop, :]

    return X, Time, Z


In [ ]:
def retransform_data(X : np.ndarray, 
                     Z : np.ndarray, 
                     Time : np.ndarray, 
                     Spec : dict, 
                     header : list, 
                     first_pred_date : datetime):  
    """  
    Retransforms the data series from stationary back to original based on Spec.Transformation  
  
    Parameters:  
        X (np.ndarray): Transformed data (stationary)  
        Z (np.ndarray): Raw data adjusted
        Time (np.ndarray): Observation periods for the time series data  
        Spec (dict): Model specification  
        header (list): Original data headers  
        first_pred_date: Date of the first prediction
  
    Returns:  
        V (np.ndarray): Retransformed data 
    """
    
    # Filter indexes that denote predictions
    prediction_idx = np.where(Time >= first_pred_date)[0]
    # Get the number of prediction periods
    T = prediction_idx.shape[0]
    # Get number of periods, series
    H = X.shape[0]
    N = X.shape[1]
    # Initialize T x N matrix filled with NaN's
    V = np.full((T, N), np.nan)  
    
    for i in range(N):  
        formula = Spec["transformation"][i]  
        freq = Spec["frequency"][i] 
        step = 1 if freq == "m" else 3  
        t1 = step  
        n = step / 12 
        assert header[i] == Spec["seriesid"][i]  
        series = Spec["seriesname"][i]

        # Apply inverse transformations based on formula  
        if formula == 'lin':  # Levels (No Transformation)  
            V[:, i] = X[:, i][prediction_idx]
        elif formula == 'chg':  # Change (Difference)
            V[0:T:step, i] = np.cumsum(X[prediction_idx[0]:H:step, i]) + Z[prediction_idx[0] - 1, i] # Assuming X[t1-1] is a last historical value
        elif formula == 'ch1': # Year over Year Change
            None
        elif formula == 'pch':  # Percent Change  
            V[0:T:step, i] = np.cumprod(1 + (X[prediction_idx[0]:H:step, i] / 100)) * Z[prediction_idx[0] - 1, i]  # Assuming X[t1-1] is a last historical value
        elif formula == 'pc1':  # Year over Year Percent Change  
            None
        elif formula == 'pca':  # Percent Change (Annual Rate)  
            V[0:T:step, i] = np.cumprod((1 + X[prediction_idx[0]:H:step, i] / 100) ** n) * Z[prediction_idx[0] - 1, i]  
        elif formula == 'log':  # Natural Log  
            V[:, i] = np.exp(X[:, i][prediction_idx])  
        else:  
            warnings.warn(f"Transformation '{formula}' not found for {series}. Using untransformed data.")  
            V[:, i] = X[:, i][prediction_idx]
    # Final matrix that contains retransformed predictions (matrix V) and raw historical data (matrix Z)
    V_final = np.full((H, N), np.nan)
    for _ in range(N):
        V_final[:,_] = np.concatenate((Z[:prediction_idx[0],_], V[:,_]))
    return V_final
# Example usage retransform_data(X, Z, Time, Spec, header, datetime(2023, 1, 1))

# Validate the algorithm

In [ ]:
# Example usage
Spec = load_spec("Spec_US_example.xls")
ds = pd.read_excel("C:\\Users\\hmagdziak001\\Desktop\\transformation\\harmonized_time_series.xlsx", engine="openpyxl", header=None, index_col=None, sheet_name="data")

X, Time, Z, header = load_data(ds, Spec)
retransformed_data = retransform_data(X, Z, Time, Spec, header, first_pred_date=datetime(2023, 1, 1))

# Validation
col_results = {}
for i in range(Z.shape[1]):
    series_name = header[i]
    # Create pd.DataFrame with column: 0 for raw data and column: 1 for retransformed data
    validation_df = pd.DataFrame([Z[:,0], retransformed_data[:,0]]).T.fillna('NA')
    # Check whether columns are not equal for some cells
    val_sum = sum(validation_df[0] != validation_df[1])
    col_results[series_name] = val_sum

print("Validation results: \n")
pd.Series(col_results)
print("\n")
print(f"Number of series incorrectly transformed: {sum(col_results.values())}")

C:\Users\hmagdziak001\AppData\Local\Temp\ipykernel_12552\2566598117.py:47: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  blocks = raw_data[block_cols].fillna(0).astype(int).values


Table 1: Model specification
              SeriesID                   SeriesName                 Units  \
0               PAYEMS           Payroll Employment  Thousands of Persons   
1               JTSJOL                 Job Openings             Thousands   
2             CPIAUCSL         Consumer Price Index                 Index   
3              DGORDER         Durable Goods Orders           $, Millions   
4                RSAFS                 Retail Sales           $, Millions   
5               UNRATE            Unemployment Rate                     %   
6                HOUST               Housing Starts    Thousands of Units   
7               INDPRO        Industrial Production                 Index   
8              DSPIC96              Personal Income   Chained $, Billions   
9              BOPTEXP                      Exports           $, Millions   
10             BOPTIMP                      Imports           $, Millions   
11             TTLCONS        Construction Spen

PAYEMS                0
JTSJOL                0
CPIAUCSL              0
DGORDER               0
RSAFS                 0
UNRATE                0
HOUST                 0
INDPRO                0
DSPIC96               0
BOPTEXP               0
BOPTIMP               0
TTLCONS               0
IR                    0
CPILFESL              0
PCEPILFE              0
PCEPI                 0
PERMIT                0
TCU                   0
BUSINV                0
IQ                    0
GACDISA066MSFRBNY     0
PCEC96                0
GACDFSA066MSFRBPHI    0
GDPC1                 0
ULCNFB                0
dtype: int64